In [ ]:
#Required Libraries

#Organize the information into tables
import pandas as pd
#Converting texts into numeric representations
from sklearn.feature_extraction.text import TfidfVectorizer
#Training our classifier
from sklearn.naive_bayes import MultinomialNB
#Separate the data for training an testing
from sklearn.model_selection import train_test_split
#Evaluate the degree of accuracy of the predictions
from sklearn.metrics import accuracy_score, classification_report


##Initial Document Dataset
The initial dataset containing 25 sample documents related to BrewKnowledge's daily operations. Each document is assigned to one of five predefined categories. This dataset serves as the initial source of information for training the classification model.

In [ ]:

# Create the initial document dataset

documents = [
    # RECIPES
    ("To prepare a vanilla latte, combine espresso, steamed milk, and vanilla syrup.", "Recipes"),
    ("Green chilaquiles are prepared with tortilla chips, green salsa, cheese, and cream.", "Recipes"),
    ("Carrot cake requires flour, grated carrots, eggs, sugar, and cinnamon.", "Recipes"),
    ("A cappuccino is made with espresso, steamed milk, and milk foam.", "Recipes"),
    ("An iced coffee is prepared with espresso, cold milk, and ice.", "Recipes"),

    # SERVICE MANUALS AND PROTOCOLS
    ("Waiters must greet customers politely and provide them with the menu.", "Service manuals and protocols"),
    ("At the end of each shift, employees must clean their workstations and organize the tables.", "Service manuals and protocols"),
    ("Waiters must communicate special customer requests to the kitchen staff.", "Service manuals and protocols"),
    ("Customer orders must be checked before being delivered to the table.", "Service manuals and protocols"),
    ("The closing procedure includes cleaning tables, checking the dining area, and reporting incidents.", "Service manuals and protocols"),

    # HYGIENE AND FOOD SAFETY POLICIES
    ("Employees must wash their hands before handling food and after using the restroom.", "Hygiene and food safety policies"),
    ("Refrigerated ingredients must be stored at safe temperatures.", "Hygiene and food safety policies"),
    ("Food allergens must be identified to prevent allergic reactions in customers.", "Hygiene and food safety policies"),
    ("Kitchen surfaces and utensils must be sanitized regularly.", "Hygiene and food safety policies"),
    ("Employees must follow food safety regulations when preparing meals.", "Hygiene and food safety policies"),

    # INVENTORY AND SUPPLIERS
    ("The manager must update the inventory whenever new coffee supplies arrive.", "Inventory and suppliers"),
    ("Purchase orders must include product quantities, prices, and supplier information.", "Inventory and suppliers"),
    ("Milk and cream stock levels must be checked every day.", "Inventory and suppliers"),
    ("The coffee shop receives coffee beans from its suppliers every week.", "Inventory and suppliers"),
    ("Expired ingredients must be removed from the inventory records.", "Inventory and suppliers"),

    # CUSTOMER SERVICE
    ("The menu provides customers with information about available drinks and meals.", "Customer service"),
    ("Customer complaints must be recorded and reviewed by the manager.", "Customer service"),
    ("The loyalty program offers rewards to frequent customers.", "Customer service"),
    ("Frequently asked questions provide information about opening hours and delivery services.", "Customer service"),
    ("Customer surveys help the coffee shop improve its products and services.", "Customer service")
]

# Convert the documents into a DataFrame
df = pd.DataFrame(documents, columns=["text", "category"])


## Dataset Expansion
The dataset expansion incorporates 25 additional sample documents, increasing the total number of examples to 50. The documents maintain an equal distribution across the five predefined categories, providing more textual variety for model training and evaluation.


In [ ]:

# Create additional documents

new_documents = [
    # RECIPES
    ("A mocha is prepared with espresso, steamed milk, and chocolate syrup.", "Recipes"),
    ("To make a vanilla frappuccino, blend coffee, milk, ice, and vanilla syrup.", "Recipes"),
    ("A breakfast sandwich contains bread, eggs, cheese, and ham.", "Recipes"),
    ("Fresh orange juice is prepared by squeezing oranges and removing the seeds.", "Recipes"),
    ("A brownie recipe includes chocolate, butter, flour, eggs, and sugar.", "Recipes"),

    # SERVICE MANUALS AND PROTOCOLS
    ("Waiters must confirm the customer's order before sending it to the kitchen.", "Service manuals and protocols"),
    ("Tables must be prepared with clean utensils before customers are seated.", "Service manuals and protocols"),
    ("Employees must follow the procedure for handling takeaway orders.", "Service manuals and protocols"),
    ("The waiter must inform the manager when a customer reports a service problem.", "Service manuals and protocols"),
    ("At the beginning of each shift, staff must check that the dining area is ready.", "Service manuals and protocols"),

    # HYGIENE AND FOOD SAFETY POLICIES
    ("Raw food must be stored separately from cooked food to prevent contamination.", "Hygiene and food safety policies"),
    ("Employees must wear clean uniforms and use appropriate protective equipment.", "Hygiene and food safety policies"),
    ("Food preparation areas must be disinfected before and after each shift.", "Hygiene and food safety policies"),
    ("Ingredients must be checked for freshness before they are used in recipes.", "Hygiene and food safety policies"),
    ("The kitchen must maintain proper temperature controls for perishable products.", "Hygiene and food safety policies"),

    # INVENTORY AND SUPPLIERS
    ("The manager records every delivery received from the coffee bean supplier.", "Inventory and suppliers"),
    ("The storeroom must maintain a record of available flour, sugar, and milk.", "Inventory and suppliers"),
    ("Suppliers must provide invoices for all purchased ingredients.", "Inventory and suppliers"),
    ("The manager reviews product quantities before placing a new purchase order.", "Inventory and suppliers"),
    ("Inventory records must include product names, quantities, and expiration dates.", "Inventory and suppliers"),

    # CUSTOMER SERVICE
    ("Customers can use the loyalty program to receive discounts on future purchases.", "Customer service"),
    ("The coffee shop provides information about its delivery options and opening hours.", "Customer service"),
    ("Customer feedback forms allow visitors to evaluate the quality of their experience.", "Customer service"),
    ("The menu includes descriptions of meals, beverages, and available ingredients.", "Customer service"),
    ("Customers can submit suggestions to improve the coffee shop's products.", "Customer service")
]

# Convert the new documents into a DataFrame
new_df = pd.DataFrame(new_documents, columns=["text", "category"])

# Add the new documents to the original dataset
df = pd.concat([df, new_df], ignore_index=True)


## Dataset Verification
Dataset verification checks the total number of documents and their distribution across the predefined categories. This step ensures that the dataset contains 50 documents, with 10 examples per category, before the training and testing process.

In [ ]:
print("Total documents:", len(df))

print("\nDocuments per category:")
print(df["category"].value_counts())

Total documents: 50

Documents per category:
category
Recipes                             10
Service manuals and protocols       10
Hygiene and food safety policies    10
Inventory and suppliers             10
Customer service                    10
Name: count, dtype: int64


## Training and Testing Data Split
The dataset is divided into two subsets: 80% for training and 20% for testing. The training subset provides the examples used by the model to learn classification patterns, while the testing subset evaluates its performance on unseen documents. The stratify parameter preserves the proportion of each category in both subsets.

In [ ]:

# Split the documents into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["category"],
    test_size=0.2,
    random_state=42,
    stratify=df["category"]
)

# Display the number of documents in each set
print("Training documents:", len(X_train))
print("Testing documents:", len(X_test))


Training documents: 40
Testing documents: 10


## Text Vectorization Using TF-IDF

TF-IDF converts textual information into numerical representations that machine learning algorithms can process. The vectorizer learns the vocabulary from the training documents and applies the same vocabulary to the testing documents, assigning numerical weights according to the importance of terms.

In [ ]:

# Initialize a new TF-IDF vectorizer
vectorizer = TfidfVectorizer()

# Learn the vocabulary and transform training documents
X_train_tfidf = vectorizer.fit_transform(X_train)

# Transform testing documents using the same vocabulary
X_test_tfidf = vectorizer.transform(X_test)

# Display the resulting dimensions
print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape:", X_test_tfidf.shape)


Training matrix shape: (40, 217)
Testing matrix shape: (10, 217)


## Model training using Naive Bayes

The Multinomial Naive Bayes algorithm is used to identify patterns in the numerical representation of the training documents. By learning the relationship between text features and category labels, the model can predict the most likely category for new documents.

In [ ]:

# Initialize the Naive Bayes classifier
model = MultinomialNB()

# Train the model using the training data
model.fit(X_train_tfidf, y_train)

print("Model training completed successfully")


Model training completed successfully


##Document classification and predicction

Document classification assigns a predicted category to each document in the testing dataset. Comparing predicted categories with their actual labels makes it possible to identify correct classifications and misclassified documents.

In [ ]:

# Predict categories for the testing documents
y_pred = model.predict(X_test_tfidf)

# Compare actual and predicted categories
results = pd.DataFrame({
    "Document": X_test.values,
    "Actual category": y_test.values,
    "Predicted category": y_pred
})

print(results.to_string(index=False))


                                                                                 Document                  Actual category               Predicted category
             Employees must wear clean uniforms and use appropriate protective equipment. Hygiene and food safety policies    Service manuals and protocols
                 Waiters must communicate special customer requests to the kitchen staff.    Service manuals and protocols    Service manuals and protocols
       Purchase orders must include product quantities, prices, and supplier information.          Inventory and suppliers          Inventory and suppliers
       The coffee shop provides information about its delivery options and opening hours.                 Customer service                 Customer service
Frequently asked questions provide information about opening hours and delivery services.                 Customer service                 Customer service
          At the beginning of each shift, staff must check that 

##Model accuracy evaluation

Accuracy measures the proportion of correctly classified documents in relation to the total number of testing documents. This metric provides an overall indication of the classifier's performance.

In [ ]:

# Calculate model accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Model accuracy:", accuracy)
print("Accuracy percentage:", accuracy * 100, "%")


Model accuracy: 0.9
Accuracy percentage: 90.0 %


##Classification performance report

The classification report provides a detailed evaluation of the model through three metrics: precision, recall, and F1-score. These measurements help assess classification performance for each predefined category and identify areas that may require improvement.

In [ ]:

# Generate the classification report
report = classification_report(y_test, y_pred, zero_division=0)

print(report)


                                  precision    recall  f1-score   support

                Customer service       1.00      1.00      1.00         2
Hygiene and food safety policies       1.00      0.50      0.67         2
         Inventory and suppliers       1.00      1.00      1.00         2
                         Recipes       1.00      1.00      1.00         2
   Service manuals and protocols       0.67      1.00      0.80         2

                        accuracy                           0.90        10
                       macro avg       0.93      0.90      0.89        10
                    weighted avg       0.93      0.90      0.89        10



## New document classification test

The new document classification test evaluates the model using a document that was not included in the original dataset or testing subset. The previously trained TF-IDF vectorizer transforms the text into numerical features, allowing the classifier to assign a predicted category.

In [ ]:

# Create a new document
new_document = [
    "The coffee supplier must deliver fresh coffee beans every Monday."
]

# Transform the new document using the trained vectorizer
new_document_tfidf = vectorizer.transform(new_document)

# Predict its category
prediction = model.predict(new_document_tfidf)

print("New document:", new_document[0])
print("Predicted category:", prediction[0])


New document: The coffee supplier must deliver fresh coffee beans every Monday.
Predicted category: Inventory and suppliers


## Implementation of Basic Semantic Search

In this section, we developed a basic semantic search component to retrieve relevant information from the coffee shop's documents. The system uses TF-IDF vectorization and cosine similarity to identify the documents that best match a natural language query.

In [ ]:
# Import cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

##Receive a natural language query

First, we defined a natural language query to represent the information an employee needs. In this example, we searched for instructions on how to prepare a vanilla latte.

In [ ]:
# Create a natural language query
query = "How to prepare a vanilla latte?"

# Display the query
print("Search query:", query)

Search query: How to prepare a vanilla latte?


##Convert the query into a vector representation using TF-IDF

We used TF-IDF to convert the 50 documents and the search query into numerical representations. This allows the system to compare the query with the documents using the same vocabulary.

In [ ]:
# Initialize a new TF-IDF vectorizer for semantic search
search_vectorizer = TfidfVectorizer()

# Transform all documents into numerical representations
documents_tfidf = search_vectorizer.fit_transform(df["text"])

# Transform the query using the same vocabulary
query_tfidf = search_vectorizer.transform([query])

# Display the resulting dimensions
print("Documents matrix shape:", documents_tfidf.shape)
print("Query matrix shape:", query_tfidf.shape)

Documents matrix shape: (50, 248)
Query matrix shape: (1, 248)


##Calculate the cosine similarity between the query and the documents

We calculated cosine similarity to measure the relationship between the query and each document. Higher similarity scores indicate that a document shares more relevant terms with the search query.

In [ ]:
# Calculate similarity between the query and all documents
similarity_scores = cosine_similarity(query_tfidf, documents_tfidf).flatten()

# Display the similarity scores
print("Similarity scores:", similarity_scores)

Similarity scores: [0.7365926  0.         0.         0.         0.         0.
 0.         0.07537932 0.07505589 0.         0.         0.
 0.06879849 0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.07958783 0.
 0.         0.         0.36700958 0.         0.         0.
 0.0705408  0.         0.         0.         0.         0.06171326
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.06189523 0.         0.06197223
 0.         0.07482861]


##Display the most relevant documents based on their similarity

Finally, we sorted the documents according to their similarity scores and displayed the five most relevant results. Each result includes the document text, its category, and its similarity score.

In [ ]:
# Sort documents by similarity score
top_indices = similarity_scores.argsort()[::-1][:5]

# Create a table with the most relevant documents
search_results = pd.DataFrame({
    "Document": df.iloc[top_indices]["text"].values,
    "Category": df.iloc[top_indices]["category"].values,
    "Similarity score": similarity_scores[top_indices]
})

# Display the search results
print("\nMost relevant documents:")
print(search_results.to_string(index=False))


Most relevant documents:
                                                                      Document                      Category  Similarity score
To prepare a vanilla latte, combine espresso, steamed milk, and vanilla syrup.                       Recipes          0.736593
    To make a vanilla frappuccino, blend coffee, milk, ice, and vanilla syrup.                       Recipes          0.367010
                     The loyalty program offers rewards to frequent customers.              Customer service          0.079588
      Waiters must communicate special customer requests to the kitchen staff. Service manuals and protocols          0.075379
          Customer orders must be checked before being delivered to the table. Service manuals and protocols          0.075056


### Additional Test: Search Example 2

To further evaluate the search component, we performed a second query related to customer complaints. This test allows us to verify whether the system can retrieve relevant information from a different document category.

In [ ]:
# Create a second natural language query
query = "How should employees handle customer complaints?"

# Transform the new query using the same vectorizer
query_tfidf = search_vectorizer.transform([query])

# Calculate cosine similarity
similarity_scores = cosine_similarity(query_tfidf, documents_tfidf).flatten()

# Sort documents by similarity score
top_indices = similarity_scores.argsort()[::-1][:5]

# Create a table with the most relevant documents
search_results = pd.DataFrame({
    "Document": df.iloc[top_indices]["text"].values,
    "Category": df.iloc[top_indices]["category"].values,
    "Similarity score": similarity_scores[top_indices]
})

# Display the second search results
print("\nSearch query:", query)
print("\nMost relevant documents:")
print(search_results.to_string(index=False))


Search query: How should employees handle customer complaints?

Most relevant documents:
                                                                          Document                         Category  Similarity score
                 Customer complaints must be recorded and reviewed by the manager.                 Customer service          0.434720
                 Employees must follow the procedure for handling takeaway orders.    Service manuals and protocols          0.163901
               Employees must follow food safety regulations when preparing meals. Hygiene and food safety policies          0.151273
      Employees must wear clean uniforms and use appropriate protective equipment. Hygiene and food safety policies          0.143069
Employees must wash their hands before handling food and after using the restroom. Hygiene and food safety policies          0.134537
